In [1]:
import os
import re
import yaml

FRONTMATTER_RE = re.compile(r'(?s)^---\s*\n(.*?)\n---\s*(.*)$')


In [2]:
def apply_priority_mapping(folder_path, mapping, priority_order=None, force=False):
    """
    Apply tag → category mapping with priority.
    If a note has no tags, assigns category "NONE".
    """
    if priority_order is None:
        priority_order = list(mapping.keys())

    priority_map = {tag.lower(): mapping[tag] for tag in mapping}

    for root, _, files in os.walk(folder_path):
        for file in files:
            if not file.endswith(".md"):
                continue

            filepath = os.path.join(root, file)
            with open(filepath, "r", encoding="utf-8") as f:
                content = f.read()

            m = FRONTMATTER_RE.match(content)
            if not m:
                continue  # skip notes without YAML
            yaml_text, body = m.groups()
            yaml_block = yaml.safe_load(yaml_text) or {}

            # normalize tags
            tags = yaml_block.get("tags", [])
            if tags is None:
                tags = []
            elif isinstance(tags, str):
                tags = [tags]
            tags = [str(t).lower().strip() for t in tags if t is not None]

            # Determine category
            if not tags:
                matched_category = "NONE"
            else:
                matched_category = None
                for tag in priority_order:
                    if tag.lower() in tags:
                        matched_category = priority_map[tag.lower()]
                        break
                if matched_category is None:
                    matched_category = "NONE"

            # Update YAML if needed
            if force or "category" not in yaml_block or not yaml_block["category"]:
                yaml_block["category"] = matched_category
                new_yaml = yaml.dump(yaml_block, sort_keys=False).strip()
                new_content = f"---\n{new_yaml}\n---\n{body}"
                with open(filepath, "w", encoding="utf-8") as f:
                    f.write(new_content)
                print(f"Updated {filepath} -> {matched_category}")
            else:
                print(f"Skipped {filepath}, already has category {yaml_block['category']}")


In [3]:
TAG_TO_CATEGORY = {
    # --- Project / workflow ---
    "drafting": "PM",
    "business": "INDUSTRY",
    "career": "INDUSTRY",
    "learning": "PM",
    "process": "PM",
    "question": "PM",
    "communication": "PM",
    "portal": "OTHER",

    # --- Machine Learning ---
    "classifier": "ML",
    "classifer": "ML",
    "regressor": "ML",
    "clustering": "ML",
    "deep_learning": "ML",
    "anomaly_detection": "ML",
    "algorithm": "CS",
    "time_series": "DS",
    "optimisation": "ML",
    "explainability": "ML",
    "evaluation": "ML",
    "modeling": "ML",
    "ml": "ML",
    "mlprocess": "ML",
    "ml_process": "ML",
    "model": "ML",
    "regularization": "ML",
    "recommendation": "ML",
    "forecasting": "DS",
    "growth_models": "ML",
        "selection": "LANG",


    # --- Data Engineering ---
    "architecture": "DE",
    "cleaning": "DE",
    "transformation": "DE",
    "governance": "DE",
    "management": "DE",
    "data_quality": "DE",
    "security": "DE",
    "database": "DE",
    "storage": "DE",
    "SQL": "DE",
    "data_governance": "DE",
    "data_pipeline": "DE",
    "data_security": "DE",
    "systems": "DE",

    # --- DevOps ---
    "devops": "DEVOPS",
    "orchestration": "DEVOPS",
    "cloud": "DEVOPS",
    "automation": "DEVOPS",

    # --- Data Analysis ---
    "querying": "DE",
    "exploration": "DS",
    "visualization": "DATA_ANALYSIS",
    "analysis": "DATA_ANALYSIS",
    "pandas": "DEVOPS",
    "preprocessing": "DATA_ANALYSIS",
    "data": "DE",
    "tool": "DEVOPS",
    "role": "DATA_ANALYSIS",

    # --- Computer Science ---
    "frontend": "DEVOPS",
    "software": "DEVOPS",
    "code_snippet": "DEVOPS",
    "system": "DEVOPS",
    "file_type": "DEVOPS",
    "test": "DEVOPS",
    "programming": "CS",
    "term": "DS",
    "documentation": "PM",
    "data_structure": "CS",
    "python": "CS",
    "graph": "CS",
    "memory_management": "DS",

    # --- Statistics ---
    "statistics": "STATISTICS",
    "math": "STATISTICS",
    "probability": "STATISTICS",

    # --- Language / NLP ---
    "GenAI": "LANG",
    "language_models": "LANG",
    "NLP": "LANG",
    "nlp": "LANG",
    "agents": "LANG",
    "prompt": "LANG",
    "flashcards": "OTHER",

    # --- Industry-specific ---
    "customer_growth": "INDUSTRY",
        "energy": "INDUSTRY",


    # --- Misc ---
    "field": "OTHER",
    "blank": "OTHER",
}


In [4]:
PRIORITY_ORDER = [
    # Machine Learning / DS
    "ml", "classifier", "classifer", "regressor", "clustering", "deep_learning", 
    "anomaly_detection", "algorithm", "time_series", "optimisation", "explainability", 
    "evaluation", "modeling", "mlprocess", "ml_process", "model", "regularization",
    "recommendation", "forecasting", "growth_models",
    
    # Statistics / Math
    "statistics", "math", "probability",
    
    # Data Analysis
    "preprocessing", "analysis", "exploration", "visualization", "pandas", 
    "data", "tool", "role", "energy",
    
    # Data Engineering
    "architecture", "cleaning", "transformation", "governance", "management", 
    "data_quality", "security", "database", "storage", "SQL", 
    "data_governance", "data_pipeline", "data_security", "systems",
    
    # CS / Software
    "software", "code_snippet", "system", "file_type", "test", "programming",
    "term", "documentation", "data_structure", "python", "graph", "memory_management",
    "frontend",
    
    # DevOps / Infrastructure
    "devops", "orchestration", "cloud", "automation",
    
    # NLP / Language / GenAI
    "GenAI", "language_models", "NLP", "nlp", "agents", "prompt", "flashcards", "selection",
    
    # Project Management / Industry
    "drafting", "business", "career", "learning", "process", "question", "communication", "portal", "customer_growth",
    
    # Misc / DS support
    "field", "blank"
]



In [5]:
apply_priority_mapping(
    folder_path="categories/uncategorised",
    mapping=TAG_TO_CATEGORY,
    priority_order=PRIORITY_ORDER,
    force=False
)


Skipped categories/uncategorised\1-on-1 Template.md, already has category INDUSTRY
Skipped categories/uncategorised\1-to-1's with a Line Manager.md, already has category INDUSTRY
Updated categories/uncategorised\AB testing.md -> DEVOPS
Updated categories/uncategorised\Accessing Gen AI generated content.md -> ML
Updated categories/uncategorised\Accuracy.md -> ML
Updated categories/uncategorised\ACF Plots.md -> NONE
Updated categories/uncategorised\ACID Transaction.md -> DE
Updated categories/uncategorised\Activation Function.md -> ML
Updated categories/uncategorised\Ada boosting.md -> DE
Skipped categories/uncategorised\Adding a database to PostgreSQL.md, already has category DE
Updated categories/uncategorised\Additive vs Multiplicative Models Time Series.md -> DS
Skipped categories/uncategorised\Addressing Multicollinearity.md, already has category STATISTICS
Updated categories/uncategorised\Addressing_Multicollinearity.py.md -> OTHER
Updated categories/uncategorised\ADF Test.md -> NO